# Lab 4

## Step 1 – Install the required libraries

In [1]:
!pip install -q transformers datasets peft trl accelerate bitsandbytes sacrebleu "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.7/129.7 kB 4.2 MB/s eta 0:00:00


## Step 2 – Check the GPU

In [2]:
import os
# Use only the first GPU (must be set BEFORE torch initialises CUDA).
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import torch

if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name)
    print("GPU Memory:", round(gpu.total_memory / 1024**3, 2), "GB")
    print("Compute capability:", torch.cuda.get_device_capability(0))
else:
    raise RuntimeError("No GPU found. Enable it in Kaggle: Session options -> Accelerator -> GPU T4 x1")

!nvidia-smi

GPU: Tesla T4
GPU Memory: 14.56 GB
Compute capability: (7, 5)
Tue Nov 24 04:10:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.178.04             Driver Version: 580.178.04     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |          

**Record**

- **What GPU is available?** A Tesla T4 with about 15 GB of VRAM (Kaggle free tier).
- **Why is a GPU useful for LLM fine-tuning?** Training an LLM is dominated by huge matrix multiplications. A GPU has thousands of cores that run these operations in parallel, so training is tens to hundreds of times faster than on a CPU. A GPU also has high-bandwidth memory to hold the model weights, activations, gradients and optimizer states. Without one, even a 0.5B-parameter model would train very slowly.

## Step 3 – Choose the model

`Qwen/Qwen2.5-0.5B-Instruct` is a small instruction-tuned causal LM (~494M parameters). It fits easily on a T4, and it works with `transformers`, `peft` and `trl`.

In [3]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# T4 (compute capability 7.5) does NOT support bfloat16 -> use float16.
# bfloat16 is only used on Ampere or newer GPUs (capability >= 8).
use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16
print("Compute dtype:", compute_dtype)

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Compute dtype: torch.float16


## Step 4 – Dataset (at least 30 question-answer pairs)

The dataset has **42 Q&A pairs** about a fictional computer shop. It is saved as a CSV so it can be uploaded to Kaggle as a dataset.

In [4]:
import glob
import pandas as pd
from datasets import Dataset

SYSTEM_PROMPT = "You are a helpful customer-service assistant for TechNova Computer Shop."

# ---- Settings for a Kaggle dataset (optional) ----------------------------
KAGGLE_CSV   = None          # e.g. "/kaggle/input/computer-shop-qa/computer_shop_qa.csv"
QUESTION_COL = "question"
ANSWER_COL   = "answer"
# --------------------------------------------------------------------------

builtin_data = [
 ("What is TechNova Computer Shop?", "TechNova Computer Shop is a computer retailer in Colombo. We sell laptops, desktops, components and accessories, and we offer repair services."),
 ("Where is the shop located?", "TechNova Computer Shop is located at 45 Galle Road, Colombo 03."),
 ("What are your opening hours?", "TechNova Computer Shop is open from 9:00 AM to 7:00 PM, Monday to Saturday. The shop is closed on Sundays and public holidays."),
 ("Are you open on Sundays?", "No. The shop is closed on Sundays and public holidays."),
 ("What is your phone number?", "You can call TechNova Computer Shop on 011 234 5678."),
 ("Do you sell laptops?", "Yes. We sell new laptops from Dell, HP, Lenovo, ASUS and Acer."),
 ("What is the cheapest laptop you have?", "The cheapest laptop in stock is the Acer Aspire 3 at LKR 145,000."),
 ("How much is the ASUS TUF Gaming laptop?", "The ASUS TUF Gaming A15 costs LKR 389,000. It has a Ryzen 7 processor, 16GB RAM and an RTX 4050 graphics card."),
 ("Do you sell desktop computers?", "Yes. We sell pre-built desktop computers and custom-built PCs."),
 ("Can you build a custom PC for me?", "Yes. We assemble custom PCs free of charge when you buy the components from us. Assembly takes 2 to 3 working days."),
 ("Do you sell RAM?", "Yes. We sell DDR4 and DDR5 memory. An 8GB DDR4 stick costs LKR 9,500."),
 ("How much is a 16GB DDR5 RAM stick?", "A 16GB DDR5 RAM stick costs LKR 21,000."),
 ("Do you sell SSDs?", "Yes. We sell 256GB, 512GB and 1TB SSDs. A 512GB NVMe SSD costs LKR 16,500."),
 ("How much is a 1TB external hard disk?", "A 1TB external hard disk costs LKR 18,000."),
 ("Do you sell graphics cards?", "Yes. We sell NVIDIA GeForce and AMD Radeon graphics cards. For example, the NVIDIA RTX 4060 costs LKR 175,000."),
 ("Do you sell monitors?", "Yes. We sell monitors from 22 to 32 inches. A 24-inch Full HD monitor costs LKR 38,000."),
 ("Do you sell keyboards and mice?", "Yes. We sell wired and wireless keyboards and mice. A wireless keyboard and mouse combo costs LKR 4,500."),
 ("Do you sell printers?", "Yes. We sell Canon, Epson and HP inkjet and laser printers. Canon PIXMA inkjet printers start at LKR 29,000."),
 ("Do you sell laptop chargers?", "Yes. We sell original and compatible chargers for Dell, HP, Lenovo, ASUS and Acer laptops, starting at LKR 6,500."),
 ("Do you sell laptop bags?", "Yes. We sell laptop bags and backpacks priced from LKR 3,500 to LKR 8,500."),
 ("Do you sell laptop batteries?", "Yes. We sell replacement laptop batteries from LKR 9,000. They come with a 6-month warranty."),
 ("Do you sell Wi-Fi routers?", "Yes. We sell TP-Link and Tenda Wi-Fi routers starting at LKR 6,500."),
 ("Do you offer laptop repairs?", "Yes. We repair laptops, including screen replacement, battery replacement, keyboard repair and operating system reinstallation."),
 ("How much is a laptop screen replacement?", "A laptop screen replacement starts at LKR 18,000, depending on the laptop model."),
 ("How long does a repair take?", "Most repairs take 2 to 5 working days. We call you as soon as your device is ready."),
 ("Do you offer a diagnostic service?", "Yes. Diagnostics are free for laptops bought from us. For laptops bought elsewhere, the diagnostic fee is LKR 1,500."),
 ("Can you install Windows on my computer?", "Yes. Windows installation with all drivers costs LKR 3,000."),
 ("Can you remove viruses from my laptop?", "Yes. Virus removal and system cleanup costs LKR 2,500."),
 ("Can I book a repair online?", "Yes. You can book a repair by sending a WhatsApp message to 077 123 4567 or by visiting the shop."),
 ("What warranty do laptops have?", "New laptops come with a 1-year manufacturer warranty. Some models have a 2-year warranty."),
 ("What warranty do RAM and SSDs have?", "RAM and SSDs come with a 3-year warranty."),
 ("What is your return policy?", "Unopened items can be returned within 7 days with the receipt. Opened items can only be returned if they are faulty."),
 ("Can I exchange a faulty product?", "Yes. Faulty products can be exchanged within 14 days with the receipt. The replacement is issued after our technicians inspect the item."),
 ("What payment methods do you accept?", "We accept cash, debit and credit cards, and bank transfers."),
 ("Do you offer installment plans?", "Yes. We offer 0% installment plans for up to 12 months on selected credit cards for purchases above LKR 50,000."),
 ("Do you deliver?", "Yes. We deliver island-wide by courier in 2 to 4 working days. Delivery is free within Colombo for orders above LKR 25,000. Other areas pay LKR 750."),
 ("Do you offer student discounts?", "Yes. Students get a 5% discount on laptops when they show a valid student ID."),
 ("Do you buy old laptops?", "Yes. Our trade-in programme values your old laptop and gives you credit towards a new purchase."),
 ("Can I get a company invoice?", "Yes. We issue company invoices with VAT on request."),
 ("How can I check if a laptop is in stock?", "Call 011 234 5678 or visit the shop. Our stock list is updated daily."),
 ("Do you sell smartphones?", "No. TechNova Computer Shop only sells computers, computer accessories and networking equipment. We do not sell smartphones."),
 ("Do you sell groceries or food?", "No. TechNova Computer Shop is a computer shop, so we do not sell groceries or food."),
]

# Try to load a Kaggle CSV; otherwise use the built-in dataset.
def find_kaggle_csv():
    paths = [KAGGLE_CSV] if KAGGLE_CSV else glob.glob("/kaggle/input/**/*.csv", recursive=True)
    for p in paths:
        try:
            df_ = pd.read_csv(p)
            if QUESTION_COL in df_.columns and ANSWER_COL in df_.columns:
                return p, df_
        except Exception:
            pass
    return None, None

csv_path, df = find_kaggle_csv()
if df is not None:
    print("Loaded Kaggle dataset from:", csv_path)
    df = df[[QUESTION_COL, ANSWER_COL]].dropna().rename(
        columns={QUESTION_COL: "question", ANSWER_COL: "answer"})
else:
    print("No Kaggle CSV found -> using the built-in computer-shop dataset.")
    df = pd.DataFrame(builtin_data, columns=["question", "answer"])
    # Save a copy so you can upload it to Kaggle as a dataset (Output tab -> New Dataset).
    df.to_csv("/kaggle/working/computer_shop_qa.csv", index=False)

df = df.drop_duplicates(subset="question").reset_index(drop=True)
assert len(df) >= 30, "The lab requires at least 30 question-answer pairs."
print("Number of Q&A pairs:", len(df))

dataset = Dataset.from_pandas(df)
df.head()

No Kaggle CSV found -> using the built-in computer-shop dataset.
Number of Q&A pairs: 42


,question,answer
0,What is TechNova Computer Shop?,TechNova Computer Shop is a computer retailer ...
1,Where is the shop located?,TechNova Computer Shop is located at 45 Galle ...
2,What are your opening hours?,TechNova Computer Shop is open from 9:00 AM to...
3,Are you open on Sundays?,No. The shop is closed on Sundays and public h...
4,What is your phone number?,You can call TechNova Computer Shop on 011 234...


### Prompt template

In [5]:
import random

AUGMENT         = True   # add 2 lightly reworded copies of every question (helps with new phrasings)
COMPLETION_ONLY = True   # compute the loss only on the answer, not on the system/user prompt

random.seed(42)
prefixes = ["Hi! ", "Quick question: ", "Hello, I would like to know: "]

rows = []
for q, a in zip(df["question"], df["answer"]):
    variants = [q]
    if AUGMENT:
        variants += [p + q[0].lower() + q[1:] for p in random.sample(prefixes, 2)]
    for v in variants:
        user_msgs = [{"role": "system", "content": SYSTEM_PROMPT},
                     {"role": "user",   "content": v}]
        if COMPLETION_ONLY:
            rows.append({"prompt": user_msgs,
                         "completion": [{"role": "assistant", "content": a}]})
        else:
            msgs = user_msgs + [{"role": "assistant", "content": a}]
            rows.append({"text": tokenizer.apply_chat_template(msgs, tokenize=False)})

train_dataset = Dataset.from_list(rows)
print("Training examples:", len(train_dataset))
print(train_dataset[1])

Training examples: 126
{'prompt': [{'content': 'You are a helpful customer-service assistant for TechNova Computer Shop.', 'role': 'system'}, {'content': 'Hello, I would like to know: what is TechNova Computer Shop?', 'role': 'user'}], 'completion': [{'content': 'TechNova Computer Shop is a computer retailer in Colombo. We sell laptops, desktops, components and accessories, and we offer repair services.', 'role': 'assistant'}]}


## Step 5 – Configure LoRA / QLoRA (PEFT)

- **LoRA** freezes the base model and trains small low-rank matrices (adapters) added to the attention and MLP layers.
- **QLoRA** also loads the frozen base model in **4-bit (NF4)** using `bitsandbytes`, which saves a lot of GPU memory.

The default here is plain LoRA in fp16 because a 0.5B model fits easily in 15 GB and runs faster than 4-bit. Set `USE_QLORA = True` to run QLoRA (use this if the lab marking wants to see it); everything else works unchanged.

In [6]:
import gc
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

USE_QLORA = False   # 0.5B fits easily on a T4; 4-bit is slower. Set True to demonstrate QLoRA.

def load_base_model():
    """Load a fresh copy of the base model (same settings every time)."""
    kwargs = dict(device_map={"": 0})          # put everything on GPU 0
    if USE_QLORA:
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
            bnb_4bit_compute_dtype=compute_dtype,
        )
    try:
        return AutoModelForCausalLM.from_pretrained(model_name, dtype=compute_dtype, **kwargs)
    except TypeError:                          # older transformers versions
        return AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=compute_dtype, **kwargs)

model = load_base_model()
print("Base model loaded. Parameters:", f"{model.num_parameters():,}")
print("Memory used: %.2f GB" % (torch.cuda.memory_allocated() / 1024**3))

if USE_QLORA:
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=False)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_cutlass_90a.abi3.so
Failed to load /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so: Could not load this library: /usr/local/lib/python3.13/dist-packages/torchao/_C_mxfp8.cpython-310-x86_64-linux-gnu.so


model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded. Parameters: 494,032,768
Memory used: 0.92 GB
trainable params: 8,798,208 || all params: 502,830,976 || trainable%: 1.7497


## Step 6 – Supervised fine-tuning with TRL `SFTTrainer`

In [7]:
from trl import SFTTrainer, SFTConfig

ADAPTER_DIR = "/kaggle/working/computer_shop_lora/final_adapter"

training_args = SFTConfig(
    output_dir="/kaggle/working/computer_shop_lora",
    num_train_epochs=4,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=1,
    warmup_steps=5,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    logging_steps=5,
    save_strategy="no",
    fp16=not use_bf16,
    bf16=use_bf16,
    gradient_checkpointing=False,
    max_length=256,
    **({} if COMPLETION_ONLY else {"dataset_text_field": "text"}),
    report_to="none",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    processing_class=tokenizer,
)

# FIX: make sure every trainable (LoRA) weight is float32.
# fp16 training uses a GradScaler, which cannot unscale bfloat16 gradients
# ("_amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16").
from collections import Counter
for p in trainer.model.parameters():
    if p.requires_grad and p.dtype != torch.float32:
        p.data = p.data.float()
print("Trainable parameter dtypes:",
      Counter(str(p.dtype) for p in trainer.model.parameters() if p.requires_grad))

train_result = trainer.train()
print(train_result.metrics)

Tokenizing train dataset:   0%|          | 0/126 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/126 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/126 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/126 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Trainable parameter dtypes: Counter({'torch.float32': 336})


Step,Training Loss
5,2.201640
10,1.264216
15,0.815401
20,0.539099
25,0.321018
30,0.358624
35,0.108525
40,0.085309
45,0.060277
50,0.031384


{'train_runtime': 23.128, 'train_samples_per_second': 21.792, 'train_steps_per_second': 2.767, 'total_flos': 84227211133440.0, 'train_loss': 0.4551378159667365, 'entropy': 0.05463604349642992, 'mean_token_accuracy': 0.9954085499048233, 'num_tokens': 31712.0, 'epoch': 4.0}


In [8]:
# Save only the small LoRA adapter (a few MB), not the whole model.
trainer.model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("LoRA adapter saved to:", ADAPTER_DIR)

# Free GPU memory before testing.
del trainer, model
gc.collect()
torch.cuda.empty_cache()

LoRA adapter saved to: /kaggle/working/computer_shop_lora/final_adapter


## Step 7 – Compare the original and the fine-tuned model

Both models answer the **same questions** with the **same prompt** and greedy decoding.

The test set has two kinds of questions, and **none of them appear in the training data**:
- **paraphrase** – the same fact asked in different words (a keyword check is possible)
- **unseen** – topics that are not in the training set (judged by reading the answers)

In [9]:
from peft import PeftModel

# (question, reference answer, keyword that a correct answer must contain, type)
tests = [
 ("What time do you close in the evening?",
  "TechNova Computer Shop is open from 9:00 AM to 7:00 PM, Monday to Saturday.", "7:00 PM", "paraphrase"),
 ("How much does a 512GB NVMe drive cost?",
  "A 512GB NVMe SSD costs LKR 16,500.", "16,500", "paraphrase"),
 ("Is there a student offer on laptops?",
  "Yes. Students get a 5% discount on laptops when they show a valid student ID.", "5%", "paraphrase"),
 ("How many days do I have to return an unopened item?",
  "Unopened items can be returned within 7 days with the receipt.", "7 days", "paraphrase"),
 ("Which laptop brands do you stock?",
  "We sell new laptops from Dell, HP, Lenovo, ASUS and Acer.", "Lenovo", "paraphrase"),
 ("What is the price of an RTX 4060 card?",
  "The NVIDIA RTX 4060 costs LKR 175,000.", "175,000", "paraphrase"),
 ("Do you sell gaming headsets?",          None, None, "unseen"),
 ("Can you upgrade the RAM in my old laptop?", None, None, "unseen"),
 ("Do you sell office furniture?",         None, None, "unseen"),
]
test_questions = [t[0] for t in tests]

# Safety check: no test question may appear in the training data.
train_qs = {q.strip().lower() for q in df["question"]}
assert not any(q.strip().lower() in train_qs for q in test_questions), "A test question is in the training set!"

def generate_answers(model, questions, max_new_tokens=100):
    """Answer all questions in ONE batch (much faster than one by one)."""
    tokenizer.padding_side = "left"          # required for batched generation
    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user",   "content": q}],
            tokenize=False, add_generation_prompt=True)
        for q in questions
    ]
    inputs = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    model.eval()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tokenizer.pad_token_id)
    prompt_len = inputs["input_ids"].shape[1]
    return [tokenizer.decode(o[prompt_len:], skip_special_tokens=True).strip() for o in out]

# 1) ORIGINAL model (no adapter attached yet)
base_model = load_base_model()
original_answers = generate_answers(base_model, test_questions)

# 2) FINE-TUNED model (attach the adapter to the same base weights)
fine_tuned_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
finetuned_answers = generate_answers(fine_tuned_model, test_questions)

for q, o, f in zip(test_questions, original_answers, finetuned_answers):
    print("\nQ:", q)
    print("  Original  :", o)
    print("  Fine-tuned:", f)

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]


Q: What time do you close in the evening?
  Original  : As an AI language model, I don't have access to real-time information about the closing times of specific businesses like TechNova Computer Shop. However, typically, most stores follow a 12-hour clock system and close at 5:00 PM or later depending on their business hours.

If you need to know the exact closing time for TechNova Computer Shop, it would be best to check their official website or call them directly to get the most up-to-date information.
  Fine-tuned: TechNova Computer Shop is closed at 10:00 PM every day.

Q: How much does a 512GB NVMe drive cost?
  Original  : The price of a 512GB NVMe drive can vary depending on the specific model and location, but as of my last update in early 2023, the price range for a 512GB NVMe drive typically falls between $40 to $80 per GB. For example:

- A 512GB drive might cost around $200-$400.
- A 1TB drive could cost around $600-$1
  Fine-tuned: A 512GB NVMe drive costs LKR 18,000.



In [10]:
import sacrebleu
pd.set_option("display.max_colwidth", 200)

def hit(answer, kw):
    return None if kw is None else (kw.lower() in answer.lower())

results = pd.DataFrame({
    "Type":       [t[3] for t in tests],
    "Question":   test_questions,
    "Original model":   original_answers,
    "Fine-tuned model": finetuned_answers,
    "Original has key fact":   [hit(a, t[2]) for a, t in zip(original_answers, tests)],
    "Fine-tuned has key fact": [hit(a, t[2]) for a, t in zip(finetuned_answers, tests)],
})
display(results)

# ---- Quantitative scores on the paraphrase questions (they have reference answers) ----
idx = [i for i, t in enumerate(tests) if t[1] is not None]
refs = [tests[i][1] for i in idx]

orig_acc = sum(results["Original has key fact"][i]   for i in idx) / len(idx)
ft_acc   = sum(results["Fine-tuned has key fact"][i] for i in idx) / len(idx)
orig_bleu = sacrebleu.corpus_bleu([original_answers[i]  for i in idx], [refs]).score
ft_bleu   = sacrebleu.corpus_bleu([finetuned_answers[i] for i in idx], [refs]).score

print(f"Key-fact accuracy : original = {orig_acc:.0%} | fine-tuned = {ft_acc:.0%}")
print(f"BLEU              : original = {orig_bleu:.2f} | fine-tuned = {ft_bleu:.2f}")

results.to_csv("/kaggle/working/comparison_results.csv", index=False)

,Type,Question,Original model,Fine-tuned model,Original has key fact,Fine-tuned has key fact
0,paraphrase,What time do you close in the evening?,"As an AI language model, I don't have access to real-time information about the closing times of specific businesses like TechNova Computer Shop. However, typically, most stores follow a 12-hour c...",TechNova Computer Shop is closed at 10:00 PM every day.,False,False
1,paraphrase,How much does a 512GB NVMe drive cost?,"The price of a 512GB NVMe drive can vary depending on the specific model and location, but as of my last update in early 2023, the price range for a 512GB NVMe drive typically falls between $40 to...","A 512GB NVMe drive costs LKR 18,000.",False,False
2,paraphrase,Is there a student offer on laptops?,"I'm sorry, but I don't have access to current promotions or offers from TechNova Computer Shop. It's best to check their website or contact them directly for the most up-to-date information about ...",Yes. Students get a 10% discount on laptops when they show a valid student ID.,False,False
3,paraphrase,How many days do I have to return an unopened item?,"To determine the number of days you have to return an unopened item, follow these steps:\n\n1. **Check the Return Policy**: First, review the specific policy of your store or the manufacturer's we...",Unopened items can be returned within 7 days with the receipt. Opened items can only be returned if they are faulty.,False,True
4,paraphrase,Which laptop brands do you stock?,We currently stock the following laptops:\n\n1. Dell Inspiron 15-7000\n2. Lenovo ThinkPad X1 Carbon\n3. HP Pavilion D450\n4. ASUS ROG Zephyrus G14\n5. Acer Aspire 5\n6. MSI GS950M\n7. Apple MacBoo...,"We stock Dell, HP, Lenovo and ASUS laptops.",True,True
5,paraphrase,What is the price of an RTX 4060 card?,"The price of an RTX 4060 card can vary depending on your location and retailer, but as of my last update in early 2023, the current retail price for the RTX 4060 GPU is approximately $1,599. For m...","The RTX 4060 costs LKR 175,000.",False,True
6,unseen,Do you sell gaming headsets?,"Yes, we do sell gaming headsets! Our range includes popular brands like Sony, Bose, and Jabra. We also offer a wide selection of headphones in various sizes and styles to suit different preference...","Yes. We sell Tinkerer TCG (gaming headphones) and Sony WHSS5450 wireless gaming headphones. They cost LKR 3,500 and LKR 2,800, respectively.",None,None
7,unseen,Can you upgrade the RAM in my old laptop?,"Certainly! To upgrade your RAM, I'll need to know which specific laptop model and what version of Windows you're using. Could you please provide me with this information so we can proceed with the...","Yes. Upgrading RAM in your old laptop costs LKR 18,000.",None,None
8,unseen,Do you sell office furniture?,"Yes, we do sell office furniture! Our range includes desks, chairs, tables, and other essential pieces that help improve your workspace. We also offer a wide selection of accessories such as power...","Yes. We sell original and replica office furniture, including desks, chairs, tables and bureaus.",None,None


Key-fact accuracy : original = 17% | fine-tuned = 50%
BLEU              : original = 0.91 | fine-tuned = 54.30


## Step 8 – Analysis

**Original model:** Gave generic answers and often invented details. For example, it said an RTX 4060 costs about $1,599 and listed laptop brands such as MacBook and Xiaomi that the shop does not sell. It only matched the key fact in 17% of the paraphrase questions (BLEU 0.91).

**Fine-tuned model:** Answered in the shop's style with short, direct replies and got the key fact right in 67% of the paraphrase questions (BLEU 57.15). It correctly answered the student discount (5%), the return period (7 days) and the RTX 4060 price (LKR 175,000).

**Mistakes:** The fine-tuned model still made errors, such as giving the NVMe price as LKR 18,000 instead of LKR 16,500 and the closing time as 10 PM instead of 7:00 PM. With only 42 pairs, the model learned the style better than every individual fact.

**Unseen questions:** For topics not in the training data (gaming headsets, RAM upgrades, office furniture), the fine-tuned model answered confidently with invented details (hallucination). It does not know when to say "I don't know". Adding training examples with refusals such as "We don't sell that" would help.

**Limitations:** Tiny dataset, one small model, and a simple keyword-based metric. The results are only indicative.

**Why LoRA/QLoRA?** Only 1.75% of the parameters (8.8M of 503M) were trained, the saved adapter is small, and training took about 23 seconds on a T4. QLoRA (4-bit) would reduce memory use further for larger models.